In [0]:
# ===========================================
# Notebook Name:
# 00_build_deck_archetypes
#
# Purpose:
# Cluster functional deck compositions into
# Pokemon-based archetype candidates.
#
# Input:
# pokemon.gold.deck_similarity
# pokemon.gold.deck_registry
# pokemon.gold.deck_pokemon_features
#
# Output:
# pokemon.gold.deck_archetypes
#
# Method:
# Agglomerative hierarchical clustering
# using precomputed Weighted Jaccard distance
# ===========================================

In [0]:
%pip install scikit-learn scipy

In [0]:
dbutils.library.restartPython()

In [ ]:
import hashlib
import uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)


DECK_SIMILARITY_TABLE = (
    "pokemon.gold.deck_similarity"
)

DECK_REGISTRY_TABLE = (
    "pokemon.gold.deck_registry"
)

DECK_FEATURES_TABLE = (
    "pokemon.gold.deck_pokemon_features"
)

DECK_ARCHETYPES_TABLE = (
    "pokemon.gold.deck_archetypes"
)

ARCHETYPE_CLUSTER_MAPPING_TABLE = (
    "pokemon.gold.archetype_cluster_mapping"
)

# A cluster smaller than this is treated as noise
# (a one-off or rogue build) rather than a recurring
# archetype, and is excluded from consideration when
# picking the best distance threshold below. Without
# this, maximizing silhouette score alone tends to
# over-fragment the metagame into many singleton
# clusters (observed: 125 clusters, 66 of them size 1,
# when no minimum was enforced), which is just as
# unusable as the opposite failure mode of one giant
# leftover cluster. Tune this if real archetypes are
# being merged away (lower it) or the leftover noise
# is still showing up as its own clusters (raise it).
MIN_ARCHETYPE_CLUSTER_SIZE = 10

# One id per notebook execution, shared with
# pipeline_run_log when this runs as a Workflow
# task. Falls back to a fresh uuid for interactive
# runs or when the initialize task is absent.
pipeline_run_id = dbutils.jobs.taskValues.get(
    taskKey="initialize_pipeline_run",
    key="pipeline_run_id",
    default=None,
    debugValue="manual-run",
)

MODEL_RUN_ID = (
    pipeline_run_id
    if pipeline_run_id
    else str(uuid.uuid4())
)

print("Input :", DECK_SIMILARITY_TABLE)
print("Input :", DECK_REGISTRY_TABLE)
print("Input :", DECK_FEATURES_TABLE)
print("Output:", DECK_ARCHETYPES_TABLE)
print("Output:", ARCHETYPE_CLUSTER_MAPPING_TABLE)
print("Model run id:", MODEL_RUN_ID)

In [0]:
registry_df = spark.table(
    DECK_REGISTRY_TABLE
)

deck_lookup_df = (
    registry_df
    .select(
        "deck_hash",
        "deck_hash_version",
        "representative_deck_code",
        "best_rank",
        "average_rank",
    )
    .dropDuplicates(["deck_hash"])
    .orderBy("deck_hash")
)

display(deck_lookup_df)

deck_count = deck_lookup_df.count()

print("Deck count:", deck_count)

In [0]:
if deck_count < 2:
    raise ValueError(
        "At least two decks are required "
        "for clustering"
    )

In [0]:
deck_lookup_pd = (
    deck_lookup_df
    .toPandas()
    .sort_values("deck_hash")
    .reset_index(drop=True)
)

deck_hashes = (
    deck_lookup_pd["deck_hash"]
    .tolist()
)

deck_index = {
    deck_hash: index
    for index, deck_hash
    in enumerate(deck_hashes)
}

display(deck_lookup_pd)

In [0]:
similarity_df = spark.table(
    DECK_SIMILARITY_TABLE
)

similarity_pd = (
    similarity_df
    .select(
        "deck_hash_a",
        "deck_hash_b",
        "weighted_jaccard_similarity",
    )
    .toPandas()
)

print(
    "Similarity pair rows:",
    len(similarity_pd),
)

In [0]:
distance_matrix = np.zeros(
    (deck_count, deck_count),
    dtype=float,
)

for row in similarity_pd.itertuples(
    index=False
):
    index_a = deck_index[
        row.deck_hash_a
    ]

    index_b = deck_index[
        row.deck_hash_b
    ]

    similarity = float(
        row.weighted_jaccard_similarity
    )

    distance = 1.0 - similarity

    distance_matrix[
        index_a,
        index_b,
    ] = distance

    distance_matrix[
        index_b,
        index_a,
    ] = distance

In [0]:
np.fill_diagonal(
    distance_matrix,
    0.0,
)

print(distance_matrix)

In [0]:
if not np.allclose(
    distance_matrix,
    distance_matrix.T,
):
    raise ValueError(
        "Distance matrix is not symmetric"
    )

if not np.allclose(
    np.diag(distance_matrix),
    0.0,
):
    raise ValueError(
        "Distance matrix diagonal is not zero"
    )

if (
    np.any(distance_matrix < 0)
    or np.any(distance_matrix > 1)
):
    raise ValueError(
        "Distance values must be "
        "between 0 and 1"
    )

print(
    "Validation passed: "
    "distance matrix is valid"
)

In [ ]:
# Distance-threshold search instead of a fixed
# max cluster count. A fixed n_clusters cap (used to
# be min(8, deck_count - 1)) forces the algorithm to
# stop merging long before the real archetype
# structure is reached: with 1478 decks, silhouette
# score kept climbing well past k=100 under the old
# cap, meaning most decks were being crammed into one
# oversized leftover cluster. The number of real
# archetypes in the metagame varies release to
# release, so instead of guessing a cap we scan cut
# heights (distances) sampled from the actual pairwise
# distance distribution and let silhouette score pick
# the best cut among them.
#
# Requiring every resulting cluster to meet
# MIN_ARCHETYPE_CLUSTER_SIZE turned out to be too
# strict: a handful of genuinely one-off "rogue"
# decks are always maximally dissimilar from
# everything else, so some singleton cluster exists
# at almost every threshold, and no threshold ever
# satisfied "all clusters >= minimum". Instead,
# silhouette is scored only on decks that land in an
# archetype-sized cluster (>= MIN_ARCHETYPE_CLUSTER_
# SIZE), the same way DBSCAN excludes noise points
# from its scoring. Decks left in undersized clusters
# still get a cluster_id in the final output further
# below; they just don't influence which threshold is
# considered best.
candidate_distance_thresholds = np.unique(
    np.quantile(
        distance_matrix[
            np.triu_indices(deck_count, k=1)
        ],
        np.linspace(0.01, 0.99, 99),
    )
)

cluster_evaluations = []

for distance_threshold in candidate_distance_thresholds:
    model = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=float(
            distance_threshold
        ),
        metric="precomputed",
        linkage="average",
    )

    labels = model.fit_predict(
        distance_matrix
    )

    cluster_sizes = (
        pd.Series(labels)
        .value_counts()
    )

    large_cluster_ids = cluster_sizes[
        cluster_sizes
        >= MIN_ARCHETYPE_CLUSTER_SIZE
    ].index

    if len(large_cluster_ids) < 2:
        continue

    scored_mask = np.isin(
        labels, large_cluster_ids
    )

    if scored_mask.sum() < 2:
        continue

    scored_labels = labels[scored_mask]

    if len(np.unique(scored_labels)) < 2:
        continue

    scored_distance_matrix = (
        distance_matrix[
            np.ix_(scored_mask, scored_mask)
        ]
    )

    score = silhouette_score(
        scored_distance_matrix,
        scored_labels,
        metric="precomputed",
    )

    cluster_evaluations.append({
        "distance_threshold": float(
            distance_threshold
        ),
        "cluster_count": len(
            np.unique(labels)
        ),
        "large_cluster_count": len(
            large_cluster_ids
        ),
        "noise_deck_count": int(
            (~scored_mask).sum()
        ),
        "silhouette_score": float(score),
        "labels": labels,
    })

    print(
        f"distance_threshold="
        f"{distance_threshold:.4f}, "
        f"clusters={len(np.unique(labels))}, "
        f"large_clusters={len(large_cluster_ids)}, "
        f"noise_decks={int((~scored_mask).sum())}, "
        f"silhouette={score:.4f}"
    )

if not cluster_evaluations:
    raise ValueError(
        "No distance threshold produced 2 or "
        "more clusters meeting "
        "MIN_ARCHETYPE_CLUSTER_SIZE="
        f"{MIN_ARCHETYPE_CLUSTER_SIZE}. "
        "Consider lowering that constant."
    )

In [ ]:
cluster_evaluation_pd = pd.DataFrame([
    {
        "distance_threshold": item[
            "distance_threshold"
        ],
        "cluster_count": item[
            "cluster_count"
        ],
        "silhouette_score": item[
            "silhouette_score"
        ],
    }
    for item in cluster_evaluations
])

display(cluster_evaluation_pd)

In [ ]:
best_evaluation = max(
    cluster_evaluations,
    key=lambda item: item[
        "silhouette_score"
    ],
)

best_distance_threshold = (
    best_evaluation[
        "distance_threshold"
    ]
)

best_cluster_count = (
    best_evaluation[
        "cluster_count"
    ]
)

best_silhouette_score = (
    best_evaluation[
        "silhouette_score"
    ]
)

print(
    "Selected distance threshold:",
    round(best_distance_threshold, 4),
)

print(
    "Selected cluster count:",
    best_cluster_count,
)

print(
    "Selected silhouette score:",
    round(
        best_silhouette_score,
        4,
    ),
)

In [ ]:
final_model = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=best_distance_threshold,
    metric="precomputed",
    linkage="average",
    compute_distances=True,
)

cluster_labels = (
    final_model.fit_predict(
        distance_matrix
    )
)

print(cluster_labels)

In [0]:
clustered_decks_pd = (
    deck_lookup_pd.copy()
)

clustered_decks_pd[
    "cluster_id"
] = cluster_labels.astype(int)

clustered_decks_pd[
    "model_name"
] = "agglomerative_weighted_jaccard"

clustered_decks_pd[
    "cluster_count"
] = best_cluster_count

clustered_decks_pd[
    "silhouette_score"
] = best_silhouette_score

clustered_decks_pd[
    "clustered_at"
] = datetime.now(
    timezone.utc
)

display(
    clustered_decks_pd.sort_values(
        [
            "cluster_id",
            "best_rank",
        ]
    )
)

In [ ]:
cluster_size_pd = (
    clustered_decks_pd
    .groupby(
        "cluster_id",
        as_index=False,
    )
    .agg(
        cluster_size=(
            "deck_hash",
            "count",
        )
    )
)

clustered_decks_pd = (
    clustered_decks_pd
    .merge(
        cluster_size_pd,
        on="cluster_id",
        how="left",
    )
)

# cluster_signature is a stable identity for "the
# same set of decks", independent of cluster_id
# (which is just an arbitrary label that can shift
# between reruns). It is the sha256 of the sorted
# deck_hash membership of the cluster, so the same
# underlying deck grouping always yields the same
# signature even if AgglomerativeClustering assigns
# it a different cluster_id next time.
cluster_signature_pd = (
    clustered_decks_pd
    .groupby("cluster_id")["deck_hash"]
    .apply(
        lambda deck_hashes: hashlib.sha256(
            ",".join(
                sorted(deck_hashes)
            ).encode("utf-8")
        ).hexdigest()
    )
    .reset_index(
        name="cluster_signature"
    )
)

clustered_decks_pd = (
    clustered_decks_pd
    .merge(
        cluster_signature_pd,
        on="cluster_id",
        how="left",
    )
)

display(
    clustered_decks_pd.sort_values(
        [
            "cluster_id",
            "best_rank",
        ]
    )
)

In [0]:
cluster_assignment_schema = StructType([
    StructField(
        "deck_hash",
        StringType(),
        False,
    ),
    StructField(
        "cluster_id",
        IntegerType(),
        False,
    ),
])

cluster_assignment_rows = [
    {
        "deck_hash": row.deck_hash,
        "cluster_id": int(
            row.cluster_id
        ),
    }
    for row in (
        clustered_decks_pd
        .itertuples(index=False)
    )
]

cluster_assignment_df = (
    spark.createDataFrame(
        cluster_assignment_rows,
        schema=cluster_assignment_schema,
    )
)

In [0]:
cluster_pokemon_usage_df = (
    spark.table(
        DECK_FEATURES_TABLE
    )
    .join(
        cluster_assignment_df,
        on="deck_hash",
        how="inner",
    )
    .groupBy(
        "cluster_id",
        "card_name",
    )
    .agg(
        F.countDistinct(
            "deck_hash"
        ).alias(
            "decks_using_pokemon"
        ),
        F.sum(
            "quantity"
        ).alias(
            "total_copies"
        ),
        F.round(
            F.avg("quantity"),
            2,
        ).alias(
            "avg_copies_when_used"
        ),
    )
)

In [0]:
cluster_sizes_df = (
    cluster_assignment_df
    .groupBy("cluster_id")
    .agg(
        F.countDistinct(
            "deck_hash"
        ).alias(
            "cluster_size"
        )
    )
)

cluster_pokemon_usage_df = (
    cluster_pokemon_usage_df
    .join(
        cluster_sizes_df,
        on="cluster_id",
        how="left",
    )
    .withColumn(
        "cluster_inclusion_rate",
        F.col(
            "decks_using_pokemon"
        )
        / F.col("cluster_size"),
    )
    .withColumn(
        "cluster_inclusion_pct",
        F.round(
            F.col(
                "cluster_inclusion_rate"
            ) * 100,
            2,
        ),
    )
)

In [0]:
display(
    cluster_pokemon_usage_df
    .orderBy(
        "cluster_id",
        F.col(
            "cluster_inclusion_rate"
        ).desc(),
        F.col(
            "total_copies"
        ).desc(),
    )
)

In [ ]:
clustered_at = datetime.now(
    timezone.utc
)

archetype_rows = []

for row in clustered_decks_pd.itertuples(
    index=False
):
    archetype_rows.append({
        "deck_hash": row.deck_hash,
        "deck_hash_version": (
            row.deck_hash_version
        ),
        "representative_deck_code": (
            row.representative_deck_code
        ),
        "cluster_id": int(
            row.cluster_id
        ),
        "cluster_signature": (
            row.cluster_signature
        ),
        "cluster_size": int(
            row.cluster_size
        ),
        "model_name": row.model_name,
        "model_run_id": MODEL_RUN_ID,
        "cluster_count": int(
            row.cluster_count
        ),
        "silhouette_score": float(
            row.silhouette_score
        ),
        "clustered_at": clustered_at,
    })

In [ ]:
archetype_schema = StructType([
    StructField(
        "deck_hash",
        StringType(),
        False,
    ),
    StructField(
        "deck_hash_version",
        StringType(),
        False,
    ),
    StructField(
        "representative_deck_code",
        StringType(),
        True,
    ),
    StructField(
        "cluster_id",
        IntegerType(),
        False,
    ),
    StructField(
        "cluster_signature",
        StringType(),
        False,
    ),
    StructField(
        "cluster_size",
        IntegerType(),
        False,
    ),
    StructField(
        "model_name",
        StringType(),
        False,
    ),
    StructField(
        "model_run_id",
        StringType(),
        False,
    ),
    StructField(
        "cluster_count",
        IntegerType(),
        False,
    ),
    StructField(
        "silhouette_score",
        DoubleType(),
        True,
    ),
    StructField(
        "clustered_at",
        TimestampType(),
        False,
    ),
])

deck_archetypes_df = (
    spark.createDataFrame(
        archetype_rows,
        schema=archetype_schema,
    )
    .withColumn(
        "updated_at",
        F.current_timestamp(),
    )
)

In [0]:
duplicate_decks_df = (
    deck_archetypes_df
    .groupBy("deck_hash")
    .count()
    .filter(
        F.col("count") > 1
    )
)

if duplicate_decks_df.count() > 0:
    display(duplicate_decks_df)

    raise ValueError(
        "Duplicate deck_hash assignments detected"
    )

In [0]:
assigned_deck_count = (
    deck_archetypes_df
    .select("deck_hash")
    .distinct()
    .count()
)

if assigned_deck_count != deck_count:
    raise ValueError(
        "Not all decks were assigned. "
        f"expected={deck_count}, "
        f"actual={assigned_deck_count}"
    )

print(
    "Validation passed: "
    "all decks assigned to one cluster"
)

In [ ]:
# deck_archetypes is rebuilt atomically via CTAS
# instead of TRUNCATE + append. TRUNCATE leaves the
# table empty for the duration of the write, so a
# failure between truncate and append would leave
# downstream consumers reading zero rows. CREATE OR
# REPLACE TABLE AS SELECT swaps the whole table in
# one atomic Delta commit: either the old data is
# fully visible, or the new data is, never neither.
# This also lets the schema evolve (e.g. dropping
# archetype_name, adding cluster_signature and
# model_run_id) without a separate ALTER TABLE.
deck_archetypes_df.createOrReplaceTempView(
    "staged_deck_archetypes"
)

spark.sql(f"""
CREATE OR REPLACE TABLE {DECK_ARCHETYPES_TABLE}
COMMENT 'Rebuilt atomically every run (CREATE OR REPLACE TABLE AS SELECT): one row per deck_hash for the latest clustering. No archetype_name column by design; see gold.v_deck_archetypes_named.'
AS SELECT * FROM staged_deck_archetypes
""")

print(
    "Gold deck_archetypes rebuilt "
    "(atomic CTAS)."
)

In [ ]:
# Register this run's clusters in
# archetype_cluster_mapping so a human can later
# attach a stable archetype_id via
# 02_review_archetype_mapping. If a cluster_signature
# was already reviewed under a previous model_run_id,
# carry that archetype_id/status/confidence forward
# instead of resetting to pending_review, since the
# signature means "the same set of decks" even though
# cluster_id and model_run_id changed.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{ARCHETYPE_CLUSTER_MAPPING_TABLE}
(
    model_run_id STRING NOT NULL,
    cluster_signature STRING NOT NULL,
    cluster_id INT NOT NULL,
    archetype_id STRING,
    mapping_status STRING NOT NULL,
    confidence DOUBLE,
    reviewed_at TIMESTAMP
)
USING DELTA
COMMENT 'One row per model_run_id x cluster_signature. Links an unstable per-run cluster_id to a stable archetype_id once reviewed.'
""")

cluster_signature_lookup_pd = (
    clustered_decks_pd[
        ["cluster_id", "cluster_signature"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

new_clusters_df = spark.createDataFrame(
    cluster_signature_lookup_pd,
    schema=StructType([
        StructField(
            "cluster_id", IntegerType(), False
        ),
        StructField(
            "cluster_signature", StringType(), False
        ),
    ]),
).withColumn(
    "model_run_id", F.lit(MODEL_RUN_ID)
)

prior_review_df = (
    spark.table(ARCHETYPE_CLUSTER_MAPPING_TABLE)
    .filter(F.col("mapping_status") == "approved")
    .groupBy("cluster_signature")
    .agg(
        F.first(
            "archetype_id", ignorenulls=True
        ).alias("prior_archetype_id"),
        F.first(
            "mapping_status", ignorenulls=True
        ).alias("prior_mapping_status"),
        F.first(
            "confidence", ignorenulls=True
        ).alias("prior_confidence"),
    )
)

staged_mapping_df = (
    new_clusters_df
    .join(
        prior_review_df,
        on="cluster_signature",
        how="left",
    )
    .select(
        "model_run_id",
        "cluster_signature",
        "cluster_id",
        F.col("prior_archetype_id").alias(
            "archetype_id"
        ),
        F.coalesce(
            F.col("prior_mapping_status"),
            F.lit("pending_review"),
        ).alias("mapping_status"),
        F.col("prior_confidence").alias(
            "confidence"
        ),
        F.lit(None).cast("timestamp").alias(
            "reviewed_at"
        ),
    )
)

staged_mapping_df.createOrReplaceTempView(
    "staged_archetype_cluster_mapping"
)

spark.sql(f"""
MERGE INTO {ARCHETYPE_CLUSTER_MAPPING_TABLE} AS target
USING staged_archetype_cluster_mapping AS source
ON  target.model_run_id = source.model_run_id
AND target.cluster_signature = source.cluster_signature

WHEN NOT MATCHED THEN INSERT (
    model_run_id,
    cluster_signature,
    cluster_id,
    archetype_id,
    mapping_status,
    confidence,
    reviewed_at
)
VALUES (
    source.model_run_id,
    source.cluster_signature,
    source.cluster_id,
    source.archetype_id,
    source.mapping_status,
    source.confidence,
    source.reviewed_at
)
""")

print(
    "archetype_cluster_mapping updated for "
    f"model_run_id={MODEL_RUN_ID}"
)

In [0]:
display(
    spark.table(
        DECK_ARCHETYPES_TABLE
    )
    .orderBy(
        "cluster_id",
        "representative_deck_code",
    )
)

In [0]:
display(
    spark.table(
        DECK_ARCHETYPES_TABLE
    )
    .groupBy("cluster_id")
    .agg(
        F.count("*").alias(
            "deck_count"
        )
    )
    .orderBy("cluster_id")
)